In [ ]:
import lightgbm as lgb
import numpy as np
import optuna
import polars as pl
from sklearn.model_selection import train_test_split
from surrogate_model.metrics import enrichment_factor, spearman_corr, top_k_recall
from surrogate_model.optuna import (
    RECALL_TOP_1_PERCENT,
    RECALL_TOP_5_PERCENT,
    make_objective,
)

In [ ]:
FEATURES = "data/sampled.parquet"
LABELS = "data/1L83.1L83:p2rank:3.output.parquet"

RANDOM_SEED = 1000

features = pl.read_parquet(FEATURES)
labels = pl.read_parquet(LABELS)
labels = labels["catalog_id", "affinity_kcal_mol"]

df = features.join(labels, on="catalog_id", how="inner")

In [ ]:
import logging

from e3fp.pipeline import fprints_from_mol
from rdkit import Chem


def generate_e3fp(sdf_str: str, bits: int = 1024):
    logging.basicConfig(level=logging.WARNING)
    logging.getLogger().setLevel(logging.WARNING)
    for logger_name in logging.root.manager.loggerDict:
        logging.getLogger(logger_name).setLevel(logging.WARNING)

    zero_vector = [0] * bits

    if not sdf_str:
        return zero_vector

    try:
        mol = Chem.MolFromMolBlock(sdf_str)
        if mol is None:
            return zero_vector

        if not mol.HasProp("_Name") or not mol.GetProp("_Name"):
            mol.SetProp("_Name", "molecule")

        fps = fprints_from_mol(mol, fprint_params={"bits": bits})

        if not fps:
            return zero_vector

        return fps[0].to_vector(sparse=False).astype(int).tolist()
    except Exception:
        return zero_vector

In [ ]:
sdf_list = df["conformer_sdf"].to_list()

In [ ]:
import concurrent.futures

from tqdm.auto import tqdm

with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(
        tqdm(
            executor.map(generate_e3fp, sdf_list, chunksize=50),
            total=len(sdf_list),
            desc="Generating fingerprints",
        )
    )

In [ ]:
df = df.with_columns(pl.Series("e3fp", results, dtype=pl.List(pl.Int64)))

In [ ]:
ALL_COLUMN_NAMES = [
    "catalog_id",
    "affinity_kcal_mol",
    "heavy_atom_count",
    "molecular_weight",
    "calculated_partition_coefficient",
    "calculated_distribution_coefficient",
    "topological_polar_surface_area",
    "hydrogen_bond_donors",
    "pka",
    "morgan_fingerprint",
    "e3fp",
]

df = df.select(ALL_COLUMN_NAMES)

In [ ]:
FEATURE_NAMES = [
    "heavy_atom_count",
    "molecular_weight",
    "calculated_partition_coefficient",
    "calculated_distribution_coefficient",
    "topological_polar_surface_area",
    "hydrogen_bond_donors",
    "pka",
]

LABEL_NAME = "affinity_kcal_mol"

x_scalars = df.select(FEATURE_NAMES).to_numpy()

x_morgan = np.array(df["morgan_fingerprint"].to_list())
x_e3fp = np.array(df["e3fp"].to_list())

y = df[LABEL_NAME].to_numpy()

In [ ]:
SAMPLE_SIZE = 25000

x = np.hstack([x_scalars, x_morgan])
# x = np.hstack([x_scalars, x_e3fp])

# x = np.hstack([x_scalars, x_morgan, x_e3fp])

x_sample = x[:SAMPLE_SIZE]
y_sample = y[:SAMPLE_SIZE]

PRIMARY_METRIC = RECALL_TOP_1_PERCENT
# PRIMARY_METRIC = RECALL_TOP_5_PERCENT

NUM_TRIALS = 30

X_train, X_test, y_train, y_test = train_test_split(
    x_sample, y_sample, test_size=0.2, random_state=RANDOM_SEED
)

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)

study.optimize(
    make_objective(
        X_train,
        y_train,
        5,
        primary_metric=PRIMARY_METRIC,
        random_seed=RANDOM_SEED,
    ),
    n_trials=NUM_TRIALS,
    show_progress_bar=True,
)

In [ ]:
best_params = study.best_params
best_params.update({"random_state": RANDOM_SEED})

final_model = lgb.LGBMRegressor(**best_params, deterministic=True, force_row_wise=True)
final_model.fit(
    X_train,
    y_train,
    eval_X=X_test,
    eval_y=y_test,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)],
)

y_pred = np.asarray(final_model.predict(X_test))

results = {
    "top_1_percent": top_k_recall(y_test, y_pred, 0.01),
    "top_5_percent": top_k_recall(y_test, y_pred, 0.05),
    "top_10_percent": top_k_recall(y_test, y_pred, 0.1),
    "spearman": spearman_corr(y_test, y_pred),
    "enrichment_factor_1_percent": enrichment_factor(y_test, y_pred, 0.01),
    "enrichment_factor_5_percent": enrichment_factor(y_test, y_pred, 0.05),
    "enrichment_factor_10_percent": enrichment_factor(y_test, y_pred, 0.1),
}

results

| dataset size | train metric | trials | fingerprint(s) | top_1_percent | top_5_percent | top_10_percent | spearman | enrichment 1 percent | enrichment 5 percent | enrichment 10 percent |
| - | - | - | - | - | - | - | - | - | - | - |
| 10000 | RECALL_TOP_1_PERCENT | 10 | morgan | 0.2 | 0.21 | 0.32 | 0.235 | 20.0 | 4.20 | 3.20 |
| 10000 | RECALL_TOP_1_PERCENT | 20 | morgan | 0.2 | 0.21 | 0.32 | 0.235 | 20.0 | 4.20 | 3.20 |
| 10000 | RECALL_TOP_1_PERCENT | 30 | morgan | 0.25 | 0.23 | 0.32 | 0.250 | 25.0 | 4.6 | 3.15 |